In [5]:
"""
Feature Engineering and Resampling for Fraud Detection - COMPLETE FIXED VERSION
Task 1 - Data Analysis and Preprocessing
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Import scikit-learn components
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

print("="*60)
print("FEATURE ENGINEERING FOR FRAUD DETECTION")
print("="*60)

# ============================================================================
# LOAD DATA
# ============================================================================

print("\nLOADING PROCESSED DATA...")

# Load processed fraud data
fraud_df = pd.read_csv('../data/processed/fraud_data_processed.csv')
credit_df = pd.read_csv('../data/processed/creditcard_processed.csv')

print(f"Fraud Data shape: {fraud_df.shape}")
print(f"Credit Card Data shape: {credit_df.shape}")

# ============================================================================
# 1. FRAUD DATA - Feature Selection and Transformation
# ============================================================================

print("\n" + "="*60)
print("PREPARING FRAUD DATA")
print("="*60)

def prepare_fraud_data(df):
    """Prepare fraud data for modeling"""
    df_prep = df.copy()
    
    # Features to keep (excluding IDs, datetime, and target)
    feature_cols = [
        'purchase_value', 'age', 'time_since_signup_hours',
        'user_transaction_count', 'time_since_last_transaction_hours',
        'avg_transaction_rate_per_hour', 'user_device_count',
        'user_browser_count', 'purchase_value_scaled',
        'purchase_hour', 'purchase_dayofweek', 'purchase_month',
        'is_weekend', 'is_night'
    ]
    
    # Check if these features exist, if not use what's available
    available_cols = [col for col in feature_cols if col in df_prep.columns]
    missing_cols = set(feature_cols) - set(available_cols)
    
    if missing_cols:
        print(f"Missing features: {missing_cols}")
        print("Using available features only...")
    
    # Add categorical features
    categorical_cols = ['source', 'browser', 'sex']
    categorical_cols = [col for col in categorical_cols if col in df_prep.columns]
    available_cols.extend(categorical_cols)
    
    print(f"\nFeatures to use: {len(available_cols)} features")
    
    # Separate features and target
    X = df_prep[available_cols].copy()
    y = df_prep['class'].copy()
    
    # One-hot encode categorical variables
    if categorical_cols:
        print(f"Encoding: {categorical_cols}")
        X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    else:
        X_encoded = X
    
    # Handle missing values
    X_encoded = X_encoded.fillna(-1)
    
    # Ensure all columns are numeric
    numeric_cols = X_encoded.select_dtypes(include=[np.number]).columns
    X_encoded = X_encoded[numeric_cols]
    
    return X_encoded, y

# Prepare fraud data
X_fraud, y_fraud = prepare_fraud_data(fraud_df)
print(f"\n✓ Fraud Data - Features: {X_fraud.shape}, Target: {y_fraud.shape}")
print(f"✓ Fraud rate: {y_fraud.mean()*100:.2f}%")

# ============================================================================
# 2. CREDIT CARD DATA - Feature Selection and Transformation
# ============================================================================

print("\n" + "="*60)
print("PREPARING CREDIT CARD DATA")
print("="*60)

def prepare_credit_data(df):
    """Prepare credit card data for modeling"""
    df_prep = df.copy()
    
    # Use V1-V28 plus scaled amount and time features
    feature_cols = [f'V{i}' for i in range(1, 29)]
    
    # Add amount and time features if they exist
    if 'Amount_scaled' in df_prep.columns:
        feature_cols.append('Amount_scaled')
    if 'Time_hours' in df_prep.columns:
        feature_cols.append('Time_hours')
    
    print(f"Features to use: {len(feature_cols)} features")
    
    # Separate features and target
    X = df_prep[feature_cols].copy()
    y = df_prep['Class'].copy()
    
    # Handle missing values
    X = X.fillna(0)
    
    return X, y

# Prepare credit card data
X_credit, y_credit = prepare_credit_data(credit_df)
print(f"\n✓ Credit Card Data - Features: {X_credit.shape}, Target: {y_credit.shape}")
print(f"✓ Fraud rate: {y_credit.mean()*100:.2f}%")

# ============================================================================
# 3. TRAIN-TEST SPLIT
# ============================================================================

print("\n" + "="*60)
print("TRAIN-TEST SPLIT")
print("="*60)

# Fraud data split
X_fraud_train, X_fraud_test, y_fraud_train, y_fraud_test = train_test_split(
    X_fraud, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud
)

# Credit card data split
X_credit_train, X_credit_test, y_credit_train, y_credit_test = train_test_split(
    X_credit, y_credit, test_size=0.2, random_state=42, stratify=y_credit
)

print(f"\nFraud Data:")
print(f"  Train: {X_fraud_train.shape}, Fraud rate: {y_fraud_train.mean()*100:.2f}%")
print(f"  Test:  {X_fraud_test.shape}, Fraud rate: {y_fraud_test.mean()*100:.2f}%")

print(f"\nCredit Card Data:")
print(f"  Train: {X_credit_train.shape}, Fraud rate: {y_credit_train.mean()*100:.2f}%")
print(f"  Test:  {X_credit_test.shape}, Fraud rate: {y_credit_test.mean()*100:.2f}%")

# ============================================================================
# 4. HANDLE CLASS IMBALANCE - Simple Random Sampling
# ============================================================================

print("\n" + "="*60)
print("CLASS IMBALANCE HANDLING - RANDOM OVERSAMPLING")
print("="*60)

def random_oversample(X, y, target_ratio=0.3):
    """Simple random oversampling for imbalanced datasets"""
    import numpy as np
    
    # Convert to numpy arrays if needed
    if isinstance(X, pd.DataFrame):
        X = X.values
    if isinstance(y, pd.Series):
        y = y.values
    
    # Separate majority and minority
    majority_idx = np.where(y == 0)[0]
    minority_idx = np.where(y == 1)[0]
    
    X_majority = X[majority_idx]
    X_minority = X[minority_idx]
    y_majority = y[majority_idx]
    y_minority = y[minority_idx]
    
    n_majority = len(X_majority)
    n_minority = len(X_minority)
    
    # Calculate how many minority samples needed
    n_target_minority = int(n_majority * target_ratio)
    
    if n_target_minority <= n_minority:
        # Just use original data
        print(f"  Using original data (no oversampling needed)")
        return X, y
    
    n_to_add = n_target_minority - n_minority
    
    # Randomly sample with replacement from minority class
    additional_idx = np.random.choice(n_minority, n_to_add, replace=True)
    X_additional = X_minority[additional_idx]
    y_additional = y_minority[additional_idx]
    
    # Combine all samples
    X_resampled = np.vstack([X, X_additional])
    y_resampled = np.hstack([y, y_additional])
    
    print(f"  Oversampled from {n_minority} to {n_target_minority} minority samples")
    
    return X_resampled, y_resampled

# Scale features
print("\nScaling features...")
scaler_fraud = StandardScaler()
X_fraud_train_scaled = scaler_fraud.fit_transform(X_fraud_train)

scaler_credit = StandardScaler()
X_credit_train_scaled = scaler_credit.fit_transform(X_credit_train)

# Apply oversampling for fraud data
print("\nFraud Data Oversampling:")
fraud_before_ratio = y_fraud_train.mean()
X_fraud_train_resampled, y_fraud_train_resampled = random_oversample(
    X_fraud_train_scaled, y_fraud_train, target_ratio=0.2
)

# Apply oversampling for credit card data
print("\nCredit Card Data Oversampling:")
credit_before_ratio = y_credit_train.mean()
X_credit_train_resampled, y_credit_train_resampled = random_oversample(
    X_credit_train_scaled, y_credit_train, target_ratio=0.2
)

# Show results
print("\n" + "="*60)
print("RESAMPLING RESULTS")
print("="*60)

print(f"\nFraud Data:")
print(f"  Before: Fraud rate = {fraud_before_ratio*100:.2f}%")
print(f"  After:  Fraud rate = {y_fraud_train_resampled.mean()*100:.2f}%")
print(f"  Shape:  {X_fraud_train_resampled.shape}")

print(f"\nCredit Card Data:")
print(f"  Before: Fraud rate = {credit_before_ratio*100:.2f}%")
print(f"  After:  Fraud rate = {y_credit_train_resampled.mean()*100:.2f}%")
print(f"  Shape:  {X_credit_train_resampled.shape}")

# ============================================================================
# 5. SAVE PREPARED DATA
# ============================================================================

print("\n" + "="*60)
print("SAVING PREPARED DATA")
print("="*60)

# Create directories if they don't exist
import os
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save fraud data
joblib.dump((X_fraud_train_resampled, y_fraud_train_resampled), 
            '../data/processed/fraud_train_resampled.pkl')
joblib.dump((X_fraud_test, y_fraud_test), 
            '../data/processed/fraud_test.pkl')
joblib.dump(scaler_fraud, '../models/fraud_scaler.pkl')
print("✓ Fraud data saved")

# Save credit card data
joblib.dump((X_credit_train_resampled, y_credit_train_resampled), 
            '../data/processed/credit_train_resampled.pkl')
joblib.dump((X_credit_test, y_credit_test), 
            '../data/processed/credit_test.pkl')
joblib.dump(scaler_credit, '../models/credit_scaler.pkl')
print("✓ Credit card data saved")

# Save feature information
feature_info = {
    'fraud_features': X_fraud.columns.tolist(),
    'credit_features': X_credit.columns.tolist(),
    'fraud_train_shape': X_fraud_train_resampled.shape,
    'credit_train_shape': X_credit_train_resampled.shape
}
joblib.dump(feature_info, '../data/processed/feature_info.pkl')
print("✓ Feature information saved")

# Save class distribution info
class_info = {
    'fraud_before': {'majority': sum(y_fraud_train == 0), 'minority': sum(y_fraud_train == 1)},
    'fraud_after': {'majority': sum(y_fraud_train_resampled == 0), 'minority': sum(y_fraud_train_resampled == 1)},
    'credit_before': {'majority': sum(y_credit_train == 0), 'minority': sum(y_credit_train == 1)},
    'credit_after': {'majority': sum(y_credit_train_resampled == 0), 'minority': sum(y_credit_train_resampled == 1)}
}
joblib.dump(class_info, '../data/processed/class_distribution.pkl')
print("✓ Class distribution info saved")

# ============================================================================
# 6. FINAL SUMMARY
# ============================================================================

print("\n" + "="*60)
print("FEATURE ENGINEERING COMPLETE!")
print("="*60)

print("\n" + "="*60)
print("SUMMARY OF PREPARED DATA")
print("="*60)

print(f"""
┌─────────────────────────────────────────────────────────────────────┐
│                      DATASET SUMMARY                               │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  FRAUD DATA:                                                       │
│    Original features: {X_fraud.shape[1]}                                  │
│    Training samples: {X_fraud_train_resampled.shape[0]}                       │
│    Test samples:     {X_fraud_test.shape[0]}                             │
│    Training fraud:   {y_fraud_train_resampled.mean()*100:.1f}%                   │
│    Test fraud:       {y_fraud_test.mean()*100:.1f}%                       │
│                                                                     │
│  CREDIT CARD DATA:                                                 │
│    Original features: {X_credit.shape[1]}                                  │
│    Training samples: {X_credit_train_resampled.shape[0]}                       │
│    Test samples:     {X_credit_test.shape[0]}                             │
│    Training fraud:   {y_credit_train_resampled.mean()*100:.1f}%                   │
│    Test fraud:       {y_credit_test.mean()*100:.1f}%                       │
│                                                                     │
│  FILES SAVED:                                                      │
│    • fraud_train_resampled.pkl                                      │
│    • fraud_test.pkl                                                │
│    • credit_train_resampled.pkl                                     │
│    • credit_test.pkl                                               │
│    • fraud_scaler.pkl                                              │
│    • credit_scaler.pkl                                             │
│    • feature_info.pkl                                              │
│    • class_distribution.pkl                                        │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
""")

print("\n✅ All data successfully prepared for modeling!")

# Verify files were saved
print("\nVerifying saved files:")
files_to_check = [
    '../data/processed/fraud_train_resampled.pkl',
    '../data/processed/fraud_test.pkl',
    '../data/processed/credit_train_resampled.pkl',
    '../data/processed/credit_test.pkl',
    '../models/fraud_scaler.pkl',
    '../models/credit_scaler.pkl'
]

for file in files_to_check:
    if os.path.exists(file):
        size = os.path.getsize(file)
        print(f"  ✓ {os.path.basename(file)} ({size:,} bytes)")
    else:
        print(f"  ✗ {os.path.basename(file)} - NOT FOUND")

FEATURE ENGINEERING FOR FRAUD DETECTION

LOADING PROCESSED DATA...
Fraud Data shape: (151112, 33)
Credit Card Data shape: (284807, 37)

PREPARING FRAUD DATA

Features to use: 17 features
Encoding: ['source', 'browser', 'sex']

✓ Fraud Data - Features: (151112, 14), Target: (151112,)
✓ Fraud rate: 9.36%

PREPARING CREDIT CARD DATA
Features to use: 30 features

✓ Credit Card Data - Features: (284807, 30), Target: (284807,)
✓ Fraud rate: 0.17%

TRAIN-TEST SPLIT

Fraud Data:
  Train: (120889, 14), Fraud rate: 9.36%
  Test:  (30223, 14), Fraud rate: 9.36%

Credit Card Data:
  Train: (227845, 30), Fraud rate: 0.17%
  Test:  (56962, 30), Fraud rate: 0.17%

CLASS IMBALANCE HANDLING - RANDOM OVERSAMPLING

Scaling features...

Fraud Data Oversampling:
  Oversampled from 11321 to 21913 minority samples

Credit Card Data Oversampling:
  Oversampled from 394 to 45490 minority samples

RESAMPLING RESULTS

Fraud Data:
  Before: Fraud rate = 9.36%
  After:  Fraud rate = 16.67%
  Shape:  (131481, 14)

